INSTALL

In [ ]:
import sys

# make sure we're on the right Python version before installing anything
REQUIRED_VERSION = (3, 11, 9)
actual_version   = sys.version_info[:3]

if actual_version != REQUIRED_VERSION:
    required_str = '.'.join(str(v) for v in REQUIRED_VERSION)
    actual_str = '.'.join(str(v) for v in actual_version)
    raise RuntimeError(
        f"Wrong Python version: expected {required_str}, got {actual_str}. "
        "check that the correct kernel is selected"
        "download from: https://www.python.org/downloads/release/python-3119/"
    )

print(f"Python {'.'.join(str(v) for v in actual_version)} ✓")

%pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu --force-reinstall
%pip install pandas numpy Pillow tqdm scikit-learn xgboost

Python 3.11.9 ✓
^C
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


IMPORTS

In [4]:
# data manipulation
import pandas as pd
import numpy as np

# deep learning - model, layers, transforms, data loading
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, TensorDataset

# image loading
from PIL import Image

# utilities
import os as os
import tqdm as tqdm

# classical ML - models and metrics
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from xgboost import XGBClassifier

PREPROCESSING

In [5]:
# paths to the raw image folders
train_image_dir = r'Data\task1_data\images\train'
test_image_dir  = r'Data\task1_data\images\test'

# fail safes
if not os.path.exists(train_image_dir):
    raise FileNotFoundError(f"Training directory not found: {train_image_dir}")

if not os.path.exists(test_image_dir):
    raise FileNotFoundError(f"Test directory not found: {test_image_dir}")

SAVE PREPROCESSED IMAGES

In [7]:
# ResNet requires 224x224 inputs.
# Mean and std values are from ImageNet (the dataset ResNet was trained on).
# Normalising to the same distribution ResNet expects ensures meaningful feature extraction.
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

def image_to_resnet(image_dir: str, output_path: str, has_labels: bool = True) -> None:
    image_tensors = []
    label_list = []
    path_list = []

    if has_labels:
        # each subfolder name is a class label
        class_names = sorted(os.listdir(image_dir))
        class_to_idx  = {class_name: idx for idx, class_name in enumerate(class_names)}

        # count total images
        total_images = sum(
            len(os.listdir(os.path.join(image_dir, cls)))
            for cls in class_names
            if os.path.isdir(os.path.join(image_dir, cls))
        )

        num_processed = 0
        for class_name in class_names:
            class_image_dir = os.path.join(image_dir, class_name)

            # skip anything that isn't a folder
            if not os.path.isdir(class_image_dir):
                continue

            for filename in os.listdir(class_image_dir):
                # skip non-image files
                if not filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                    continue

                image_path = os.path.join(class_image_dir, filename)
                img = Image.open(image_path).convert("RGB")

                image_tensors.append(transform(img))
                label_list.append(class_to_idx[class_name])
                path_list.append(image_path)

                num_processed += 1
                print(f'{num_processed}/{total_images}', end='\r')

        torch.save({
            'tensors': torch.stack(image_tensors),
            'labels': torch.tensor(label_list),
            'paths': path_list,
            'class_to_idx': class_to_idx
        }, output_path)

    else:
        # test folder is flat so no labels
        image_filenames = [
            f for f in os.listdir(image_dir)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ]
        total_images = len(image_filenames)

        for image_idx, filename in enumerate(image_filenames, start=1):
            image_path = os.path.join(image_dir, filename)
            img = Image.open(image_path).convert("RGB")

            image_tensors.append(transform(img))
            path_list.append(image_path)

            print(f'{image_idx}/{total_images}', end='\r')

        torch.save({
            'tensors': torch.stack(image_tensors),
            'paths': path_list
        }, output_path)

    print(f"\nSaved {len(image_tensors)} images to {output_path}")

image_to_resnet(train_image_dir, r'Data\task1_data\t1_train.pt', has_labels=True)
image_to_resnet(test_image_dir, r'Data\task1_data\t1_test.pt',  has_labels=False)

3750/3750
Saved 3750 images to Data\task1_data\t1_train.pt
1250/1250
Saved 1250 images to Data\task1_data\t1_test.pt


RESNET FEATURE EXTRACTION

In [8]:
# load ResNet50 with pretrained ImageNet weights
resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

# strip the final classification head, we only want the feature extractor part.
# this leaves a model that outputs a 2048-d vector per image instead of class scores.
resnet = nn.Sequential(*list(resnet.children())[:-1])
resnet.eval()  # inference only, no gradient tracking needed

def extract_resnet_features(pt_path: str, output_path: str) -> None:
    saved_data = torch.load(pt_path, weights_only=False)
    image_tensors = saved_data['tensors']   # shape: (N, 3, 224, 224)
    total_images = len(image_tensors)
    feature_list = []

    with torch.no_grad():
        for image_idx, image_tensor in enumerate(image_tensors, start=1):
            # ResNet expects a batch dimension: (3,224,224) -> (1,3,224,224)
            feature_vector = resnet(image_tensor.unsqueeze(0))  # output: (1, 2048, 1, 1)

            # remove the batch and spatial dims: (1, 2048, 1, 1) -> (2048,)
            feature_vector = feature_vector.squeeze().numpy()

            feature_list.append(feature_vector)
            print(f'{image_idx}/{total_images}', end='\r')

    # stack all individual feature vectors into one (N, 2048) matrix
    features = np.array(feature_list)

    # merge the feature matrix into the original data dict and save
    output_data = {**saved_data, 'features': features}
    torch.save(output_data, output_path)
    print(f"\nExtracted features shape: {features.shape} -> saved to {output_path}")

extract_resnet_features(r'Data\task1_data\t1_train.pt', r'Data\task1_data\t1_train_resnet.pt')
extract_resnet_features(r'Data\task1_data\t1_test.pt',  r'Data\task1_data\t1_test_resnet.pt')

3750/3750
Extracted features shape: (3750, 2048) -> saved to Data\task1_data\t1_train_resnet.pt
1250/1250
Extracted features shape: (1250, 2048) -> saved to Data\task1_data\t1_test_resnet.pt


CLASSIFICATION MODELS

In [9]:
"""
Classification model definitions. We try four different approaches:
  1. Linear Classifier  (logistic regression)
  2. SVM                (support vector machine)
  3. XGBoost            (gradient boosted trees)
  4. MLP                (multi-layer perceptron, a small neural network)

All train_* functions share the same interface:
    X_train, X_test : np.ndarray of shape (n_samples, n_features)
    y_train, y_test : np.ndarray of shape (n_samples,) with integer class labels
They return: (fitted_model, metrics_dict)
"""

def _to_numpy(X: np.ndarray, y: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    # convert to plain numpy arrays with consistent dtypes
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y, dtype=np.int64)
    return X, y


def _compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float | dict[int, float]]:
    # count how many samples belong to each class
    class_ids, class_counts = np.unique(y_true, return_counts=True)

    # spotting class imbalance
    class_balance = {}
    for class_id, count in zip(class_ids, class_counts):
        class_balance[int(class_id)] = round(100 * count / len(y_true), 1)

    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred, average='weighted'),
        'precision': precision_score(y_true, y_pred, average='weighted'),
        'recall': recall_score(y_true, y_pred, average='weighted'),
        'class_balance': class_balance,
    }


class MLP(nn.Module):
    def __init__(
        self,
        input_dim: int,
        num_classes: int,
        activation = nn.GELU,
        hidden: tuple[int, ...] = (64, 32),
        dropout:float = 0.0,
    ):
        super().__init__()
        layers = []
        layer_input_size = input_dim

        for layer_output_size in hidden:
            # fully connected layer: maps from the previous size to this layer's size
            layers.append(nn.Linear(layer_input_size, layer_output_size))

            # add the activation function after each linear layer
            if activation == nn.GELU:
                # tanh approximation is faster and works well in practice
                layers.append(nn.GELU(approximate="tanh"))
            else:
                try:
                    # a few activations like PReLU take the layer size as an argument
                    layers.append(activation(layer_output_size))
                except TypeError:
                    # standard activations (ReLU, SELU and some others) take no arguments
                    layers.append(activation())
                    
            # we only really look into GELU, RELU and SELU, but just incase we can try others

            # dropout randomly zeros out a fraction of neurons during training,
            # which forces the network to not rely on any single neuron. This reduces overfitting
            if dropout > 0:
                layers.append(nn.Dropout(dropout))

            layer_input_size = layer_output_size

        # final layer maps from the last hidden layer to one score per class
        layers.append(nn.Linear(layer_input_size, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def train_linear_classifier(
    X_train: np.ndarray, y_train: np.ndarray,
    X_test: np.ndarray, y_test: np.ndarray,
) -> tuple[LogisticRegression, dict[str, float | dict[int, float]]]:
    X_train, y_train = _to_numpy(X_train, y_train)
    X_test, y_test = _to_numpy(X_test,  y_test)

    # high iteration cap so it always converges
    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_train, y_train)

    predictions = clf.predict(X_test)
    return clf, _compute_metrics(y_test, predictions)


def train_svm(
    X_train: np.ndarray, y_train: np.ndarray,
    X_test: np.ndarray, y_test: np.ndarray,
    kernel: str = 'linear',
) -> tuple[SVC, dict[str, float | dict[int, float]]]:
    X_train, y_train = _to_numpy(X_train, y_train)
    X_test, y_test = _to_numpy(X_test, y_test)

    # decision_function_shape='ovr' = one-vs-rest strategy for multi-class
    clf = SVC(kernel=kernel, decision_function_shape='ovr')
    clf.fit(X_train, y_train)

    predictions = clf.predict(X_test)
    return clf, _compute_metrics(y_test, predictions)


def train_xgboost(
    X_train: np.ndarray, y_train: np.ndarray,
    X_test: np.ndarray, y_test: np.ndarray,
) -> tuple[XGBClassifier, dict[str, float | dict[int, float]]]:
    X_train, y_train = _to_numpy(X_train, y_train)
    X_test, y_test = _to_numpy(X_test, y_test)

    # multi:softmax = multi-class output, mlogloss = multi-class log loss evaluation
    clf = XGBClassifier(objective='multi:softmax', eval_metric='mlogloss')
    clf.fit(X_train, y_train)

    predictions = clf.predict(X_test)
    return clf, _compute_metrics(y_test, predictions)


def train_mlp(
    X_train: np.ndarray, y_train: np.ndarray,
    X_test: np.ndarray, y_test:  np.ndarray,
    hidden: tuple[int, ...]  = (64, 32),
    activation: type[nn.Module] = nn.GELU,
    dropout: float = 0.0,
    epochs: int = 200,
    lr: float = 1e-3,
    device: str | None = None,
    seed: int = 42,
) -> tuple[MLP, dict[str, float | dict[int, float]]]:
    X_train, y_train = _to_numpy(X_train, y_train)
    X_test, y_test = _to_numpy(X_test, y_test)

    # i have a GPU, but you may not. so the net will run on GPU if available, otherwise CPU. it will be slower on CPU, but should still work.
    # by default, i set the notebook to download CPU version only, it will be slow :( unfortch
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    num_classes = len(np.unique(y_train))

    torch.manual_seed(seed)  # makes weight init and dropout reproducible

    # convert numpy arrays to PyTorch tensors and move them to the chosen device
    X_train_tensor = torch.tensor(X_train.astype(np.float32), device=device)
    y_train_tensor = torch.tensor(y_train, device=device)
    X_test_tensor = torch.tensor(X_test.astype(np.float32), device=device)
    y_test_tensor = torch.tensor(y_test, device=device)

    # DataLoader handles batching and shuffling the training data each epoch
    train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=256, shuffle=True)

    # build the MLP and put it on the device
    model = MLP(
        input_dim = X_train_tensor.shape[1],
        num_classes = num_classes,
        activation  = activation,
        hidden = hidden,
        dropout = dropout,
    ).to(device)

    # Adam with a small weight decay to discourage huge weights, avoid shitty models that overfit
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    # cross-entropy loss is standard for multi-class classification
    criterion = nn.CrossEntropyLoss()

    # early stopping state: track the best validation loss and save those weights
    best_val_loss = float("inf")
    best_model_weights = None
    epochs_without_improvement = 0

    for epoch in range(epochs):

        # ── training pass ──────────────────────────────────────────────────────
        model.train()  # training mode: enables dropout
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()                              # clear gradients from last step
            batch_loss = criterion(model(X_batch), y_batch)   # forward pass + compute loss
            batch_loss.backward()                              # backprop: compute gradients
            optimizer.step()                                   # update weights using those gradients

        # ── validation pass ────────────────────────────────────────────────────
        model.eval()  # eval mode: disables dropout so predictions are deterministic
        with torch.no_grad():
            val_loss = criterion(model(X_test_tensor), y_test_tensor).item()

        # save weights if this is the best we've seen
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_weights = model.state_dict()
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            # stop early if val loss hasn't improved in 10 consecutive epochs
            if epochs_without_improvement >= 10:
                break

    # restore the weights from the best epoch before running final predictions
    model.load_state_dict(best_model_weights)
    model.eval()

    with torch.no_grad():
        # argmax picks the class with the highest score for each sample
        test_predictions = model(X_test_tensor).argmax(dim=1).cpu().numpy()

    return model, _compute_metrics(y_test, test_predictions)

MODEL EXECUTION

In [10]:
# load the ResNet features we extracted earlier
train_data = torch.load(r'Data\task1_data\t1_train_resnet.pt', weights_only=False)
test_data = torch.load(r'Data\task1_data\t1_test_resnet.pt',  weights_only=False)

# X = feature matrix, y = class labels (conventional ML naming)
X: np.ndarray = train_data['features']  # (3750, 2048) - all labelled images
y: np.ndarray = train_data['labels'].numpy()  # (3750,) - integer class label per image
X_test: np.ndarray = test_data['features']  # (1250, 2048) - unlabelled submission images (no y_test)

print(f'X: {X.shape}, y: {y.shape}')
print(f'X_test (submission): {X_test.shape}')
print(f'Classes: {train_data["class_to_idx"]}')

# split labelled data into train and validation sets
# stratify=y ensures both splits have the same class proportions as the full set.
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f'\nX_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_val: {X_val.shape}, y_val: {y_val.shape}')

X: (3750, 2048), y: (3750,)
X_test (submission): (1250, 2048)
Classes: {'bird': 0, 'butterfly': 1, 'cat': 2, 'deer': 3, 'dog': 4, 'elephant': 5, 'frog': 6, 'horse': 7, 'sheep': 8, 'spider': 9}

X_train: (3000, 2048), y_train: (3000,)
X_val: (750, 2048), y_val: (750,)


In [11]:
from sklearn.model_selection import StratifiedKFold

# ── Hyperparameters ───────────────────────────────────────────────────────────
SVM_KERNEL     = 'rbf'            # 'linear' | 'rbf' | 'poly' | 'sigmoid'

MLP_HIDDEN     = (1024, 512, 256)
MLP_ACTIVATION = nn.ReLU          # nn.GELU | nn.ReLU | nn.SELU
MLP_DROPOUT    = 0.0
MLP_EPOCHS     = 200
MLP_LR         = 1e-3
# ─────────────────────────────────────────────────────────────────────────────

# stratified k-fold: each fold has the same class distribution as the full dataset.
# we evaluate on all labelled samples to get the most reliable estimate.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def _cv_report(name: str, fold_results: list[dict]) -> None:
    """Print mean ± std across all CV folds for each metric."""
    print(f'--- {name} (5-fold CV) ---')
    
    for metric_name in ('accuracy', 'f1', 'precision', 'recall'):
        # collect this metric's value from every fold, then summarise
        fold_values = [fold[metric_name] for fold in fold_results]
        mean_value = np.mean(fold_values)
        std_value = np.std(fold_values)
        print(f'  {metric_name:<9}: {mean_value:.4f} ± {std_value:.4f}')
        
    # class balance is identical across stratified folds - print it once from fold 0
    print('  balance  :')
    for class_id, percentage in fold_results[0]['class_balance'].items():
        print(f'    class {class_id}: {percentage}%')
    print()

print('Evaluating models with 5-fold CV...\n')

# ── Linear Classifier ────────────────────────────────────────────────────────
fold_results = []
for train_idx, val_idx in cv.split(X, y):
    _, metrics = train_linear_classifier(X[train_idx], y[train_idx], X[val_idx], y[val_idx])
    fold_results.append(metrics)
_cv_report('Linear Classifier', fold_results)

# ── SVM ──────────────────────────────────────────────────────────────────────
fold_results = []
for train_idx, val_idx in cv.split(X, y):
    _, metrics = train_svm(X[train_idx], y[train_idx], X[val_idx], y[val_idx], kernel=SVM_KERNEL)
    fold_results.append(metrics)
_cv_report(f'SVM  kernel={SVM_KERNEL}', fold_results)

# ── XGBoost ──────────────────────────────────────────────────────────────────
fold_results = []
for train_idx, val_idx in cv.split(X, y):
    _, metrics = train_xgboost(X[train_idx], y[train_idx], X[val_idx], y[val_idx])
    fold_results.append(metrics)
_cv_report('XGBoost', fold_results)

# ── MLP ──────────────────────────────────────────────────────────────────────
fold_results = []
for train_idx, val_idx in cv.split(X, y):
    _, metrics = train_mlp(
        X[train_idx], y[train_idx],
        X[val_idx],   y[val_idx],
        hidden     = MLP_HIDDEN,
        activation = MLP_ACTIVATION,
        dropout    = MLP_DROPOUT,
        epochs     = MLP_EPOCHS,
        lr         = MLP_LR,
    )
    fold_results.append(metrics)
_cv_report(f'MLP  hidden={MLP_HIDDEN}  dropout={MLP_DROPOUT}', fold_results)

Evaluating models with 5-fold CV...

--- Linear Classifier (5-fold CV) ---
  accuracy : 0.9187 ± 0.0126
  f1       : 0.9187 ± 0.0129
  precision: 0.9198 ± 0.0132
  recall   : 0.9187 ± 0.0126
  balance  :
    class 0: 10.0%
    class 1: 10.0%
    class 2: 10.0%
    class 3: 10.0%
    class 4: 10.0%
    class 5: 10.0%
    class 6: 10.0%
    class 7: 10.0%
    class 8: 10.0%
    class 9: 10.0%

--- SVM  kernel=rbf (5-fold CV) ---
  accuracy : 0.9163 ± 0.0081
  f1       : 0.9166 ± 0.0078
  precision: 0.9188 ± 0.0073
  recall   : 0.9163 ± 0.0081
  balance  :
    class 0: 10.0%
    class 1: 10.0%
    class 2: 10.0%
    class 3: 10.0%
    class 4: 10.0%
    class 5: 10.0%
    class 6: 10.0%
    class 7: 10.0%
    class 8: 10.0%
    class 9: 10.0%

--- XGBoost (5-fold CV) ---
  accuracy : 0.8968 ± 0.0187
  f1       : 0.8966 ± 0.0188
  precision: 0.8978 ± 0.0185
  recall   : 0.8968 ± 0.0187
  balance  :
    class 0: 10.0%
    class 1: 10.0%
    class 2: 10.0%
    class 3: 10.0%
    class 4: 10.

HYPER PARAM SELECTION

In [12]:
from sklearn.model_selection import cross_val_score, ParameterGrid
from itertools import product

# ── SVM ───────────────────────────────────────────────────────────────────────
# per-kernel grid so each kernel only sees the params that actually affect it:
#   linear  : only C matters (C controls the margin width - lower = softer margin)
#   rbf     : C + gamma (gamma controls how far a single training point's influence reaches)
#   sigmoid : C + gamma
#   poly    : C + gamma + degree (degree is the polynomial degree)
svm_param_grid = [
    {'kernel': ['linear'],   'C': [0.01, 0.1, 1, 10, 100]},
    {'kernel': ['rbf'],      'C': [0.1, 1, 10, 100],  'gamma': ['scale', 'auto', 0.01, 0.001]},
    {'kernel': ['sigmoid'],  'C': [0.1, 1, 10, 100],  'gamma': ['scale', 'auto']},
    {'kernel': ['poly'],     'C': [0.1, 1, 10],        'gamma': ['scale', 'auto'], 'degree': [2, 3, 4]},
]
svm_configs = list(ParameterGrid(svm_param_grid))

print(f'SVM search ({len(svm_configs)} configs, 5-fold CV each):')
best_svm_f1     = -1
best_svm_params = {}

svm_bar = tqdm.tqdm(svm_configs, desc='SVM configs')
for svm_params in svm_bar:
    clf        = SVC(decision_function_shape='ovr', **svm_params)
    cv_scores  = cross_val_score(clf, X_train, y_train, cv=5, scoring='f1_weighted', n_jobs=-1)
    mean_score = cv_scores.mean()

    if mean_score > best_svm_f1:
        best_svm_f1     = mean_score
        best_svm_params = svm_params

    svm_bar.set_postfix({
        'kernel': svm_params['kernel'],
        'f1':     f'{mean_score:.4f}',
        'best':   f'{best_svm_f1:.4f}',
    })

print(f'\n  best SVM -> {best_svm_params}  f1={best_svm_f1:.4f}')

# ── MLP ───────────────────────────────────────────────────────────────────────
mlp_grid = {
    'hidden': [
        # 2 layers
        (512, 256),
        (1024, 512),
        # 3 layers
        (1024, 512, 256),
        # 4 layers
        (1024, 512, 256, 128),
        (512, 256, 128, 64),
        # 5 layers
        (1024, 512, 256, 128, 64),
        # 6 layers
        (1024, 512, 256, 128, 64, 32),
        # 7 layers
        (1024, 512, 256, 128, 64, 32, 16),
    ],
    'activation': [nn.GELU, nn.ReLU, nn.SELU],
    'dropout':    [0.0, 0.2, 0.4],
    'lr':         [1e-3, 3e-4],
}

# generate every combination of the above hyperparameters
all_configs = list(product(mlp_grid['hidden'], mlp_grid['activation'], mlp_grid['dropout'], mlp_grid['lr']))

print(f'\nMLP grid search ({len(all_configs)} configs):')
best_mlp_f1     = -1
best_mlp_config = {}

mlp_bar = tqdm.tqdm(all_configs, desc='MLP configs')
for hidden, activation, dropout, lr in mlp_bar:
    _, metrics = train_mlp(X_train, y_train, X_val, y_val,
                           hidden=hidden, activation=activation,
                           dropout=dropout, lr=lr)
    val_f1 = metrics['f1']

    if val_f1 > best_mlp_f1:
        best_mlp_f1     = val_f1
        best_mlp_config = dict(hidden=hidden, activation=activation, dropout=dropout, lr=lr)

    mlp_bar.set_postfix({
        'act':    activation.__name__,
        'drop':   dropout,
        'lr':     lr,
        'f1':     f'{val_f1:.4f}',
        'best':   f'{best_mlp_f1:.4f}',
    })

print(f'\n  best MLP -> {best_mlp_config}  f1={best_mlp_f1:.4f}')

SVM search (47 configs, 5-fold CV each):


SVM configs: 100%|██████████| 47/47 [06:24<00:00,  8.17s/it, kernel=poly, f1=0.2294, best=0.9153]   



  best SVM -> {'C': 1, 'degree': 2, 'gamma': 'scale', 'kernel': 'poly'}  f1=0.9153

MLP grid search (144 configs):


MLP configs:  90%|█████████ | 130/144 [07:37<02:24, 10.33s/it, act=GELU, drop=0.2, lr=0.0003, f1=0.8665, best=0.9223]c:\Users\TheRe\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
MLP configs: 100%|██████████| 144/144 [09:11<00:00,  3.83s/it, act=SELU, drop=0.4, lr=0.0003, f1=0.8906, best=0.9223]


  best MLP -> {'hidden': (512, 256), 'activation': <class 'torch.nn.modules.activation.ReLU'>, 'dropout': 0.2, 'lr': 0.0003}  f1=0.9223


SUBMISSION

In [ ]:
from sklearn.model_selection import train_test_split as _split_for_early_stopping

# ── Pick your model and fill in the best params from the hyper param search ──
SUBMISSION_MODEL = 'svm'   # 'linear' | 'svm' | 'xgboost' | 'mlp'

# SVM - paste best params from hyper param search
SVM_SUB_PARAMS = {'kernel': 'rbf', 'C': 10, 'gamma': 'scale'}

# MLP - paste best config from hyper param search
MLP_SUB_HIDDEN     = (512, 128)
MLP_SUB_ACTIVATION = nn.ReLU
MLP_SUB_DROPOUT    = 0.2
MLP_SUB_LR         = 3e-4
MLP_SUB_EPOCHS     = 200
# ─────────────────────────────────────────────────────────────────────────────

print(f'Training {SUBMISSION_MODEL} on full labelled set ({len(X)} samples)...')

if SUBMISSION_MODEL == 'linear':
    submission_clf = LogisticRegression(max_iter=1000)
    submission_clf.fit(np.asarray(X, dtype=np.float64), np.asarray(y, dtype=np.int64))
    predictions = submission_clf.predict(np.asarray(X_test, dtype=np.float64))

elif SUBMISSION_MODEL == 'svm':
    submission_clf = SVC(decision_function_shape='ovr', **SVM_SUB_PARAMS)
    submission_clf.fit(np.asarray(X, dtype=np.float64), np.asarray(y, dtype=np.int64))
    predictions = submission_clf.predict(np.asarray(X_test, dtype=np.float64))

elif SUBMISSION_MODEL == 'xgboost':
    submission_clf = XGBClassifier(objective='multi:softmax', eval_metric='mlogloss')
    submission_clf.fit(np.asarray(X, dtype=np.float64), np.asarray(y, dtype=np.int64))
    predictions = submission_clf.predict(np.asarray(X_test, dtype=np.float64))

elif SUBMISSION_MODEL == 'mlp':
    # MLP needs a small val set purely for early stopping - hold out 5%.
    # we still effectively train on almost all labelled data (95%).
    X_sub_train, X_sub_val, y_sub_train, y_sub_val = _split_for_early_stopping(
        X, y, test_size=0.05, random_state=42, stratify=y
    )
    submission_model, _ = train_mlp(
        X_sub_train, y_sub_train,
        X_sub_val,   y_sub_val,
        hidden     = MLP_SUB_HIDDEN,
        activation = MLP_SUB_ACTIVATION,
        dropout    = MLP_SUB_DROPOUT,
        epochs     = MLP_SUB_EPOCHS,
        lr         = MLP_SUB_LR,
    )
    submission_model.eval()
    with torch.no_grad():
        # convert X_test to a tensor and run it through the trained model
        test_features_tensor = torch.tensor(np.asarray(X_test, dtype=np.float32))
        predictions = submission_model(test_features_tensor).argmax(dim=1).numpy()

else:
    raise ValueError(f'Unknown model: {SUBMISSION_MODEL!r}')

# extract image IDs from file paths - just the filename without the extension
image_ids = [
    os.path.splitext(os.path.basename(file_path))[0]
    for file_path in test_data['paths']
]

# build the submission dataframe and write to CSV
submission = pd.DataFrame({'image_id': image_ids, 'class_id': predictions})
submission.to_csv('submission.csv', index=False)

print(f'Saved {len(submission)} predictions -> submission.csv')